In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Reading data

In [26]:
df_cases = pd.read_excel("../resources/weekly_varicella_hun.xlsx", sheet_name="weekly_cases")
df_age_structured_cases = pd.read_excel("../resources/age_structured_vzv_hun.xlsx", sheet_name="varicella")
df_susceptibles = pd.read_excel("../resources/number_of_susceptibles_hun.xlsx", sheet_name="s0")
df_population = pd.read_excel("../resources/age_structured_population.xlsx", sheet_name="population")
df_births = pd.read_excel("../resources/births_hun.xlsx", sheet_name="births")
df_deaths = pd.read_excel("../resources/age_structured_deaths_hun.xlsx", sheet_name="deaths", index_col=0)
df_vaccines = pd.read_excel("../resources/vaccine_coverage_hun.xlsx", sheet_name="vaccines")
df_contacts = pd.read_excel("../resources/contact_mtx_hun.xlsx", sheet_name="contacts", index_col=0)

## 2. Convert dates and set indices

### 2.1 Varicella data

#### 2.1.1 Weekly cases

In [35]:
# "2020_1" → split year and week
df_cases[["Year", "WeekNum"]] = df_cases["Week"].str.split("_", expand=True)
df_cases["Year"] = df_cases["Year"].astype(int)
df_cases["WeekNum"] = df_cases["WeekNum"].astype(int)

# ISO week -> date (the first day of the week, Monday)
df_cases["Date"] = pd.to_datetime(df_cases["Year"].astype(str) + df_cases["WeekNum"].astype(str) + "1", format="%G%V%u")

# set date index
df_weekly_cases = df_cases.set_index("Date").sort_index()

start_of_vaccination = pd.Timestamp("2019-09-01")
start_of_age_structured_date = pd.Timestamp("2005-01-01")

detection_rate_before = 0.4
detection_rate_after = 0.4

weekly_cases = df_weekly_cases["Cases"].copy()

weekly_cases.loc[weekly_cases.index < start_of_vaccination] *= 1 / detection_rate_before
weekly_cases.loc[weekly_cases.index >= start_of_vaccination] *= 1 / detection_rate_after

# fill missing values
full_index = pd.date_range(
    start=start_of_age_structured_date,
    end=weekly_cases.index.max(),
    freq="W-MON"
)
weekly_cases_full = weekly_cases.reindex(full_index)
weekly_cases_filled = weekly_cases_full.interpolate(method="linear")

before_vaccination_mask = weekly_cases_filled.index < start_of_vaccination
with_vaccination_mask = weekly_cases_filled.index >= start_of_vaccination

#### 2.1.2 Age-structured data

In [27]:
age_groups = df_age_structured_cases.iloc[:, 0].astype(str).tolist()
years = df_age_structured_cases.columns[1:].astype(int).tolist()
annual_cases = df_age_structured_cases.set_index(df_age_structured_cases.columns[0])
annual_cases.index = (
    annual_cases.index
    .map(str)          # minden elem string
    .str.strip()       # szóközök eltávolítása
    .str.replace("–", "-", regex=False)   # unicode dash → hyphen
)

In [16]:
def weekly_proportions_for_year(year):
    mask = weekly_cases.index.year == year
    year_data = weekly_cases[mask]
    total = year_data.sum()
    return year_data / total

In [33]:
weekly_age_structured = {}

for age in age_groups:
    weekly_age_structured[age] = {}
    for year in years:
        annual_value = annual_cases.loc[age, year]
        props = weekly_proportions_for_year(year)
        weekly_age_structured[age][year] = props * annual_value

test


In [36]:
i = weekly_cases_filled.values
T = len(i)
A = len(age_groups)

### 2.2 Birth and death data; vaccination coverage

In [34]:
weekly_index = weekly_cases_filled.index

# Yearly data in dictionary format
births_yearly = dict(zip(df_births["Year"], df_births["Cases"]))
deaths_yearly = dict(zip(df_deaths["Year"], df_deaths["Cases"]))
v1_yearly = dict(zip(df_vaccines["Year"], df_vaccines["1st dose"]))
v2_yearly = dict(zip(df_vaccines["Year"], df_vaccines["2nd dose"]))

# Empty series for the weekly data
weekly_births = pd.Series(index=weekly_index, dtype=float)
weekly_deaths = pd.Series(index=weekly_index, dtype=float)
weekly_v1 = pd.Series(index=weekly_index, dtype=float)
weekly_v2 = pd.Series(index=weekly_index, dtype=float)

# Filling
for date in weekly_index:
    year = date.year
    weekly_births.loc[date] = births_yearly[year] / 52
    # weekly_births.loc[date] = births_yearly[year] * birth_rates[date.weekofyear]
    weekly_deaths.loc[date] = deaths_yearly[year] / 52
    #weekly_deaths.loc[date] = deaths_yearly[year] * death_rates[date.weekofyear]
    birth_date_of_vaccinated = date + pd.DateOffset(months=-1)
    if date < start_of_vaccination:
        weekly_v1.loc[date] = 0
        weekly_v2.loc[date] = 0
    elif (date >= start_of_vaccination) & (date < pd.Timestamp("2020-01-01")):
        weekly_v1.loc[date] = v1_yearly[year] / (52 - start_of_vaccination.weekofyear)
        weekly_v2.loc[date] = 0
    elif date < start_of_vaccination + pd.DateOffset(months=18):
        weekly_v1.loc[date] = v1_yearly[year] / 52
        weekly_v2.loc[date] = 0
    else:
        weekly_v1.loc[date] = v1_yearly[year] / 52
        weekly_v2.loc[date] = v2_yearly[year] / 52


birth_mtx = np.zeros((T,A))
birth_mtx[:, 0] = weekly_births

KeyError: 'Year'

### 2.3 Initial values

#### 2.3.1 Number of susceptibles

In [19]:
df_susceptibles.iloc[:, 0] = (
    df_susceptibles.iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.replace("–", "-", regex=False)
)
df_susceptibles.iloc[:, 1] = pd.to_numeric(df_susceptibles.iloc[:, 1], errors="coerce")
S0 = dict(zip(df_susceptibles.iloc[:, 0], df_susceptibles.iloc[:, 1]))
df_S0 = df_susceptibles.set_index(df_susceptibles.columns[0])
S0_vector = df_S0.iloc[:, 0].values.astype(float)

#### 2.3.2 Population size

In [20]:
df_population.iloc[:,0] = (
    df_population.iloc[:, 0]
    .astype(str)
    .str.strip()
    .str.replace("–", "-", regex=False)
)
df_population.iloc[:, 1] = pd.to_numeric(df_population.iloc[:, 1], errors="coerce")
N0 = dict(zip(df_population.iloc[:, 0], df_population.iloc[:, 1]))
df_N0 = df_population.set_index(df_population.columns[0])
N0_vector = df_N0.iloc[:, 0].values.astype(float)

### 2.4 Contact matrix

In [21]:
df_contacts.index = (
    df_contacts.index.astype(str)
    .str.strip()
    .str.replace("–", "-", regex=False)
)

df_contacts.columns = (
    df_contacts.columns.astype(str)
    .str.strip()
    .str.replace("–", "-", regex=False)
)

contacts0 = df_contacts.values

## 3. Calculating the number of susceptibles

In [25]:
V1_efficacy = 0.81
V2_efficacy = 0.92
v1 = weekly_v1.values
v2 = weekly_v2.values

S_scen0 = np.zeros((A, T)) ## scen0 = actual scenario;
S_scen0[:, 0] = S0_vector
N = np.zeros((A, T))
N[:, 0] = N0_vector
m = np.zeros((A, A, T))

for t in range(1, T):
    Ni = N[:, t-1].reshape(-1, 1)
    Nj = N[:, t-1].reshape(1, -1)
    cm = (contacts0 * Ni + contacts0.T * Nj) / (2 * Ni)

    # Update the population
    N[t] = N[t-1] + weekly_births.iloc[t-1] - weekly_deaths.iloc[t-1]

C:\Users\CSURITA\AppData\Local\Temp\ipykernel_4176\3942791333.py:18: RuntimeWarning: divide by zero encountered in divide
  cm = (contacts0 * Ni + contacts0.T * Nj) / (2 * Ni)
C:\Users\CSURITA\AppData\Local\Temp\ipykernel_4176\3942791333.py:18: RuntimeWarning: invalid value encountered in divide
  cm = (contacts0 * Ni + contacts0.T * Nj) / (2 * Ni)

KeyboardInterrupt

